# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Dataset/rokomari_books/Rokomari Recommendation Dataset/Datasets
# %cd /content/drive/MyDrive/Rokomari Recommendation Dataset

/content/drive/.shortcut-targets-by-id/1SdeIcOv6xY-c8y2UcMwPLYx_BaB0zdXx/Rokomari Recommendation Dataset/Datasets


In [ ]:
!ls

'Copy of rokomari_bn_books_modeling_ft.ipynb'   rokomari_books_only_bangla_v2.csv
 corrected_language.csv			       'Rokomari RS task-sheet.gsheet'
'Data Analysis Report.gdoc'		        rokomari_v2.csv
 mixed_title_rokomari_v2.csv		        rokomari_v2.ipynb
 rokomari_book_data.csv			       'scraping log.txt'
 rokomari_book_data_v2.csv		        wrong_language_url.txt


# 2. Imports

In [ ]:
!git clone https://github.com/facebookresearch/fastText.git
!cd fastText
!sudo pip install fastText
! # or :
# ! sudo python setup.py install

Cloning into 'fastText'...
remote: Enumerating objects: 3998, done.
remote: Counting objects: 100% (1057/1057), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 3998 (delta 922), reused 884 (delta 856), pack-reused 2941 (from 1)
Receiving objects: 100% (3998/3998), 8.30 MiB | 9.36 MiB/s, done.
Resolving deltas: 100% (2529/2529), done.
Updating files: 100% (520/520), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fastText: filename=fasttext-0.9.3-cp310-cp310-linux_x86_64.whl size=4296188 sha256=357448f0a33c27b9da1cd967e89cef5aa35b81248d5d9090d326ed04de434f6c
  Stored in directory: /root/.cache/pip/wheels/0d/a2/00/81db54d3e6a8199b829d58e02cec2ddb20ce3e59fad

In [ ]:
!pip install rapidfuzz

In [ ]:
# import necessary packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rapidfuzz import process, fuzz

In [ ]:
# Deep learning packages

from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import fasttext

# 3. Data load

In [ ]:
# load the titles coreected dataset

df = pd.read_csv("rokomari_books_only_bangla_v2.csv")

In [ ]:
# show the head of the dataset
df.head()

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,অমানুষিক,bn,bn,পশ্চিমবঙ্গের বই,bn
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,নির্বাচিত গল্প সংকলন,bn,bn,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",bn
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,তিমুর ও তার দলবল,bn,bn,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",bn
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0,চেক ডিসঅনার মামলার সহজ ভাষ্য,bn,bn,ব্যাংকিং এন্ড কমার্স ল,bn
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,অ্যাডভোকেসি/বিচার আইন,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn,bn,"অ্যাডভোকেসি, বিচার আইন",bn


## 3.1 Essential portion of the Data

In [ ]:
# let's just create a dataframe that we are going to use

# we'll take author, bangla_title, categories_fiexd, also in that order
# -------------------------------------------------------------
df = df.loc[:, ['author', 'bangla_title', 'categories_fixed']]
# -------------------------------------------------------------


# also, merge the author, bangla_title and categories_fixed to form a
# combined feature
# -----------------------------------------------------------------------------------------------
df['combined_feature'] = df['author'] + ' ' + df['bangla_title'] + ' ' + df['categories_fixed']
# df['combined_feature'] = df['bangla_title'] + ' ' + df['categories_fixed']
# -----------------------------------------------------------------------------------------------

In [ ]:
# see how the dataframe looks now
df.head()

,author,bangla_title,categories_fixed,combined_feature
0,মনোরঞ্জন ব্যাপারী,অমানুষিক,পশ্চিমবঙ্গের বই,মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই
1,লু স্যুন,নির্বাচিত গল্প সংকলন,"পশ্চিমবঙ্গের বই, সমকালীন গল্প","লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই,..."
2,আর্কাদি গাইদার,তিমুর ও তার দলবল,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...
3,মোঃ কাইছার হামিদ,চেক ডিসঅনার মামলার সহজ ভাষ্য,ব্যাংকিং এন্ড কমার্স ল,মোঃ কাইছার হামিদ চেক ডিসঅনার মামলার সহজ ভাষ্য ...
4,মোঃ কাইছার হামিদ,হাইকোর্ট এক্সাম ফর্মুলা,"অ্যাডভোকেসি, বিচার আইন",মোঃ কাইছার হামিদ হাইকোর্ট এক্সাম ফর্মুলা অ্যাড...


In [ ]:
# let's just look at an example from the newly created
# combined_feature column

df.loc[0, 'combined_feature']

'মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই'

### 3.1.2 Remove special symbols

In [ ]:
# remove special symbols like :ঃ,- etc
# --------------------------------------------
import re
def remove_special_symbols(text):
    return re.sub(r'[ঃ:,-]', '', str(text))
# --------------------------------------------


In [ ]:
# apply the special character removal funciton on the
# combined_feature column, also split items of the
# combined_feature column and put the split version to
# the tokenized_feature

# -----------------------------------------------------------------------------------
df['combined_feature'] = df.loc[:, 'combined_feature'].apply(remove_special_symbols)
# df['tokenized_feature'] = df.loc[:, 'combined_feature'].apply(simple_preprocess)
df['tokenized_feature'] = df['combined_feature'].apply(lambda x: x.split())
# -----------------------------------------------------------------------------------


In [ ]:
# see the head again! :p
df.head()

,author,bangla_title,categories_fixed,combined_feature,tokenized_feature
0,মনোরঞ্জন ব্যাপারী,অমানুষিক,পশ্চিমবঙ্গের বই,মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই,"[মনোরঞ্জন, ব্যাপারী, অমানুষিক, পশ্চিমবঙ্গের, বই]"
1,লু স্যুন,নির্বাচিত গল্প সংকলন,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই ...,"[লু, স্যুন, নির্বাচিত, গল্প, সংকলন, পশ্চিমবঙ্গ..."
2,আর্কাদি গাইদার,তিমুর ও তার দলবল,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...,"[আর্কাদি, গাইদার, তিমুর, ও, তার, দলবল, পশ্চিমব..."
3,মোঃ কাইছার হামিদ,চেক ডিসঅনার মামলার সহজ ভাষ্য,ব্যাংকিং এন্ড কমার্স ল,মো কাইছার হামিদ চেক ডিসঅনার মামলার সহজ ভাষ্য ব...,"[মো, কাইছার, হামিদ, চেক, ডিসঅনার, মামলার, সহজ,..."
4,মোঃ কাইছার হামিদ,হাইকোর্ট এক্সাম ফর্মুলা,"অ্যাডভোকেসি, বিচার আইন",মো কাইছার হামিদ হাইকোর্ট এক্সাম ফর্মুলা অ্যাডভ...,"[মো, কাইছার, হামিদ, হাইকোর্ট, এক্সাম, ফর্মুলা,..."


In [ ]:
df.isna().sum()

,0
author,0
bangla_title,0
categories_fixed,0
combined_feature,0
tokenized_feature,0
author_count,0


### 3.2.2 get author counts

In [ ]:
# Merge the author counts (how many times an authors name occured in the df)
# with the original df
# ------------------------------------------------------------------------------------
df = df.merge(df['author'].value_counts().reset_index(), on='author', how='inner').\
        rename(columns={'count': 'author_count'})
# ------------------------------------------------------------------------------------

In [ ]:
print(f"Shape: {df.shape}")

Shape: (204572, 6)


In [ ]:
# tokenized_books = df['tokenized_feature'].tolist()


# 4. Fasttext

## 4.1 Functions

In [ ]:
def average_word_vectors(words, model, vocabulary, num_features):
    feature_vector = np.zeros((num_features,), dtype="float64")
    nwords = 0.

    for word in words:
        if word in vocabulary:
            nwords = nwords + 1.
            feature_vector = np.add(feature_vector, model.wv[word])

    if nwords:
        feature_vector = np.divide(feature_vector, nwords)

    return feature_vector

# Function to compute average word vectors for all books
def averaged_word_vectorizer(corpus, model, num_features):
    vocabulary = set(model.wv.index_to_key)
    features = [average_word_vectors(tokenized_sentence, model, vocabulary, num_features) for tokenized_sentence in corpus]
    return np.array(features)


def get_book_vector(words, model):
    # Get vectors for each word, ignore words not in the model's vocabulary

    word_vectors = [model.wv[word] for word in words if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)  # Return zero vector if no words match
    return np.mean(word_vectors, axis=0)

### 4.1.2 get vector embeddings of books for corresponding models

In [ ]:
def get_average_embedding(df, user_input, model):
    user_input = user_input.lower()

    matching_books = df[
        (df['bangla_title'].str.contains(user_input)) |
        (df['author'].str.contains(user_input)) |
        (df['categories_fixed'].str.contains(user_input))
    ]

    if not matching_books.empty:
        # Get embeddings for all matching books
        embeddings = np.array([model.get_sentence_vector(' '.join(row))
                               for row in matching_books[['bangla_title', 'author', 'categories_fixed']].values])
        # Return the average embedding
        return np.mean(embeddings, axis=0)
    else:
        return None



class RecommendBooks:
    def __init__(self, dataframe, model):
        self.df = dataframe
        self.model = model

    # get the embedding for the word averaged by
    # all the words in the input
    # -----------------------------------------------------------------------
    def get_phrase_embedding(self, phrase):

        """
        returns the average embedding of a passed phrase
        args:
            phrase: string
            model: word embedding model
        """
        words = phrase.split()  # Tokenize the phrase into words
        word_vectors = [self.model.get_word_vector(word) for word in words]
        return sum(word_vectors) / len(word_vectors)  # Average the word vectors
    # -----------------------------------------------------------------------


    # Function for book recommendation
    # -----------------------------------------------------------------------------
    def recommend_books(self, user_input, top_n=10):
        """
        Recommends books based on a user input.
        args:
            df: [Pandas DataFrame object] [required]
            user_input: string [required]
            model: word embedding model [required]
            top_n: int [default 10]

        returns:
            [Pandas DataFrame object]
        """
        # Get the embedding for the user's input
        target_embedding = self.get_phrase_embedding(user_input)

        # Compute similarity scores between the target embedding and all book embeddings
        similarities = cosine_similarity(target_embedding.reshape(1, -1),
                                        np.stack(df['embedding'].values))[0]

        # Get the indices of the most similar books
        similar_books_indices = similarities.argsort()[-top_n:][::-1]

        # Return the most similar books
        return df.iloc[similar_books_indices][['author', 'bangla_title', 'categories_fixed']]
        # -----------------------------------------------------------------------------

## 4.2. Train

In [ ]:
df['combined_feature'].to_csv('books_data.txt', index=False, header=False)

In [ ]:
data_path = 'books_data.txt'
model_algo = 'skipgram'
minn = 2
maxn = 6
dim = 100
ws = 4
epoch = 5
lr = 0.03

In [ ]:
# wv_model = Word2Vec(sentences=tokenized_books, vector_size=100, window=4, min_count=1, workers=4)
# fstxt = FastText(vector_size=100, window=4, min_count=1, workers=4)

# fstxt = FastText(vector_size=100, window=3, workers=4)

ft_model = fasttext.train_unsupervised(data_path,
                                       model=model_algo,
                                       minn=minn,
                                       maxn=maxn,
                                       dim=dim,
                                       ws=ws,
                                       epoch=epoch,
                                       lr=lr)

# fstxt.build_vocab(tokenized_books)
# fstxt.train(tokenized_books, total_examples=len(tokenized_books), epochs=30)


In [ ]:
ft_model.save_model('fasttext_model_v2')

In [ ]:
df['embedding'] = df['combined_feature'].apply(lambda x: ft_model.get_sentence_vector(x))

In [ ]:
recommendations = RecommendBooks(df, ft_model)

In [ ]:
recommendations.recommend_books('Bank', top_n=20)

,author,bangla_title,categories_fixed
23039,প্লাজমিড প্লাস পাবলিকেশন্স,বিইউপি ফুটেজ কনসেপ্ট বুক এন্ড কোয়েশ্চেন ব্যাংক...,বিশ্ববিদ্যালয় ভর্তি প্রস্তুতি
42142,সংশপ্তক পাবলিকেশন্স সম্পাদক,সংশপ্তক জাহাঙ্গীরনগর বিশ্ববিদ্যালয় ভর্তি গাইড ...,ব্যবসায় শিক্ষা ভর্তি প্রস্তুতি
23055,প্লাজমিড প্লাস পাবলিকেশন্স,বিইউপি ফুটেজ কনসেপ্ট বুক এন্ড কোশ্চেন ব্যাংক এ...,বিশ্ববিদ্যালয় ভর্তি প্রস্তুতি
7294,ছায়ামঞ্চ পাবলিকেশন্স,সামিট ভার্সিটি সলূশন B D ইউনিট,মানবিক বিভাগ ভর্তি প্রস্তুতি
23060,মো আলআমিন ছোটন,বিইউপি ফুটেজ কনসেপ্ট বুক এন্ড কোয়েশ্চেন ব্যাংক...,বিশ্ববিদ্যালয় ভর্তি প্রস্তুতি
80991,বীকন পাবলিকেশন্স সম্পাদক,ব্যাংকার্স সম্পাদিত Real and Mock Viva For Ban...,ব্যাংক চাকরি নিয়োগ প্রস্তুতি
74210,টিম ডিসপারসিভ এনারজি,ঢাবি এ ইউনিট রিটেন এনার্জি বুক,বিজ্ঞান বিভাগ ভর্তি প্রস্তুতি
69717,হালদা পাবলিকেশন্স,হালদা চট্টগ্রাম বিশ্ববিদ্যালয় মডেল টেস্ট বাণিজ...,ব্যবসায় শিক্ষা ভর্তি প্রস্তুতি
6772,ছায়ামঞ্চ পাবলিকেশন্স,জাহাঙ্গীরনগর স্পেশাল প্রশ্ন ব্যাংক বিজ্ঞান বিভ...,বিজ্ঞান বিভাগ ভর্তি প্রস্তুতি
23040,প্লাজমিড প্লাস পাবলিকেশন্স,BUP ফুটেজ ফাইনাল মডেল টেস্ট All Unit,বিশ্ববিদ্যালয় ভর্তি প্রস্তুতি


In [ ]:
df['book_vector_FT'] = df['tokenized_feature'].apply(lambda x: get_book_vector(x, fstxt)) # get vector embedding based on the models

In [ ]:
df

,author,bangla_title,categories_fixed,combined_feature,tokenized_feature,author_count,book_vector_FT
0,লু স্যুন,নির্বাচিত গল্প সংকলন,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই ...,"[লু, স্যুন, নির্বাচিত, গল্প, সংকলন, পশ্চিমবঙ্গ...",3,"[-0.92316365, -0.09355732, 0.92529637, 1.46180..."
1,আর্কাদি গাইদার,তিমুর ও তার দলবল,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...,"[আর্কাদি, গাইদার, তিমুর, ও, তার, দলবল, পশ্চিমব...",5,"[-0.7975182, 0.07547871, 1.0636704, -1.0945891..."
2,মতি নন্দী,নায়কের প্রবেশ ও প্রস্থান,"পশ্চিমবঙ্গের বই, সমকালীন উপন্যাস",মতি নন্দী নায়কের প্রবেশ ও প্রস্থান পশ্চিমবঙ্গে...,"[মতি, নন্দী, নায়কের, প্রবেশ, ও, প্রস্থান, পশ্চ...",12,"[-1.3726246, -0.2834918, 1.125387, -0.75571537..."
3,উৎপলকুমার বসু,হাঁস চলার পথ,"পশ্চিমবঙ্গের বই, বাংলা কবিতা",উৎপলকুমার বসু হাঁস চলার পথ পশ্চিমবঙ্গের বই বাং...,"[উৎপলকুমার, বসু, হাঁস, চলার, পথ, পশ্চিমবঙ্গের,...",3,"[-1.2224694, 0.5827744, 0.39400467, -0.0939196..."
4,দিলরুবা হাসান,অনুভূতির ছোঁয়া,"বয়স যখন ৪-৮, শিক্ষামূলক",দিলরুবা হাসান অনুভূতির ছোঁয়া বয়স যখন ৪৮ শিক্...,"[দিলরুবা, হাসান, অনুভূতির, ছোঁয়া, বয়স, যখন, ...",1,"[-0.058502227, -0.20464875, 1.1152658, 1.73978..."
...,...,...,...,...,...,...,...
95598,আ শ ম বাবর আলী,আবোল তাবোল ছড়া,"বয়স যখন ৪-৮, বাংলা ছড়া",আ শ ম বাবর আলী আবোল তাবোল ছড়া বয়স যখন ৪৮ বাংল...,"[আ, শ, ম, বাবর, আলী, আবোল, তাবোল, ছড়া, বয়স, য...",31,"[1.0065821, -0.21709919, 0.9430484, 0.97506005..."
95599,বিলাল হোসাইন নূরী,কুসুম ফোটার বেলা,"বয়স যখন ৮-১২, গল্প",বিলাল হোসাইন নূরী কুসুম ফোটার বেলা বয়স যখন ৮১২...,"[বিলাল, হোসাইন, নূরী, কুসুম, ফোটার, বেলা, বয়স,...",2,"[0.30260718, 0.9993237, 0.6470882, 1.0822091, ..."
95600,সতীনাথ ভাদুড়ী,সতীনাথ ভাদুড়ীঃ শতবার্ষিকী রচনা সংকলন ২য় খণ্ড,"পশ্চিমবঙ্গের বই, রচনাসমগ্র, সংকলন",সতীনাথ ভাদুড়ী সতীনাথ ভাদুড়ী শতবার্ষিকী রচনা সং...,"[সতীনাথ, ভাদুড়ী, সতীনাথ, ভাদুড়ী, শতবার্ষিকী, র...",9,"[-0.37723944, -0.58072984, 0.66124725, 2.78810..."
95601,চিত্তরঞ্জন ঘোষাল,গীতা সংগ্রহ: শ্রীমদভগবদ্গীতাসহ ৩৫টি গীতা একখণ্...,হিন্দু ধর্মীয় বই,চিত্তরঞ্জন ঘোষাল গীতা সংগ্রহ শ্রীমদভগবদ্গীতাসহ...,"[চিত্তরঞ্জন, ঘোষাল, গীতা, সংগ্রহ, শ্রীমদভগবদ্গ...",4,"[-0.70602226, 0.14308749, 1.057601, -0.3846019..."


In [ ]:
fstxt.wv.most_similar("হুমায়ুন আহমেদ",topn=5)

[('হুমায়ুন', 0.8042830228805542),
 ('হুমায়রা', 0.7370495200157166),
 ('হুমায়ূন', 0.7030653953552246),
 ('হুমায়ূন', 0.6970828771591187),
 ('আহমেদ', 0.6459429860115051)]

In [ ]:
# Function to get similar books based on input text
def ft_similar_books(input_text, model=fstxt, topn=20):

    input_tokens = input_text.split() # Tokenize the input text
    input_vector = get_book_vector(input_tokens, model) #avg word vector for tokens

    book_vectors = np.vstack(df['book_vector_FT'].values)     # cosine similarity between input vector and all book vectors
    similarities = cosine_similarity([input_vector], book_vectors)[0]

    # Get top N most similar books
    df['similarity_score'] = similarities
    results = df[['author', 'bangla_title', 'categories_fixed', 'similarity_score']].sort_values(by='similarity_score', ascending=False).head(topn)

    return results

In [ ]:
ft_similar_books('দেয়াল') # get similar books based on titles

,author,bangla_title,categories_fixed,similarity_score
38338,কামরুন নাহার সুমী,লাল বেলুন,চিত্রনাট্য,0.608574
13191,কাজল শাহনেওয়াজ,গতকাল লাল,সমকালীন গল্প,0.604969
75761,তাহেরা সুমনা,ছোট্ট লাল মুরগী,"ব্যঙ্গ, রম্যরচনা",0.604370
67832,জয়নাল আবেদীন জুয়েল,বাঘের দেশে শেয়াল রাজা,বাংলা কবিতা,0.598215
67046,নাসির আহমেদ কাবুল,হলুদ বৃন্ত লাল গোলাপ,সমকালীন উপন্যাস,0.597972
10016,আহমেদ রিয়াজ,সাদা হাতি কালো গরু,ছড়া,0.588831
18767,নাজনীন তৌহিদ,লাল বড়ুজান,সমকালীন উপন্যাস,0.575028
25106,সুহাসিনী,লাল চশমা কালো চশমা,সমকালীন উপন্যাস,0.570111
42417,সঞ্জয় কর,লাল সবুজের ফেরিওয়ালা,মুক্তিযুদ্ধের গল্প,0.569799
1506,জয়নুল টিটো,বিউটিবোনে লাল পিপড়া,সমকালীন গল্প,0.567529


In [ ]:
ft_similar_books('এই পরিটা লাল টুকটুক ওই পরিটা নীল	')

,author,bangla_title,categories_fixed,similarity_score
7782,শাহনেওয়াজ চৌধুরী,এই পরিটা লাল টুকটুক ওই পরিটা নীল,শিশু-কিশোর গল্প,0.825469
86378,তৃধা আনিকা,লাল রঙের নীল বৃষ্টি,সমকালীন উপন্যাস,0.737284
23840,সুহাসিনী,লাল চশমা কালো চশমা,সমকালীন উপন্যাস,0.718412
31037,মিরাজ আহম্মেদ,লাল নীল স্বপ্ন,বাংলা কবিতা,0.712255
63442,নিলীশা নীল,আজ এই মেঘলা আসমান,সমকালীন উপন্যাস,0.706737
64308,সুমন তুরহান,এই পাথির প্রবাহ এই ঋতুবতী মেঘ,বাংলা কবিতা,0.702439
21505,রোস্তম মল্লিক,নীল নরকে মৌন আকাশ,বাংলা কবিতা,0.701253
33304,মরিয়ম বেগম,লাল নীল সবুজ,বাংলা কবিতা,0.689663
22966,নাজনীন তৌহিদ,লাল বড়ুজান,সমকালীন উপন্যাস,0.685443
47166,আমজাদ হোসাইন,নীল রং প্রজাপতি,সমকালীন উপন্যাস,0.680086


In [ ]:
text = "পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস" #based on genres
ft_similar_books(text)

,author,bangla_title,categories_fixed,similarity_score
21779,ষষ্ঠীপদ চট্টোপাধ্যায়,দুইটি উপন্যাস কিশোর,"পশ্চিমবঙ্গের বই, উপন্যাস",0.897013
35157,সঞ্জীব চট্টোপাধ্যায়,তিন কিশোর,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",0.857706
85347,কল্যাণ সুন্দরম,বুনুর গল্প: বাংলা উপন্যাস,"পশ্চিমবঙ্গের বই, উপন্যাস",0.854745
58,সুনীল গঙ্গোপাধ্যায় নীললোহিত,বাছাই দশটি উপন্যাস,"পশ্চিমবঙ্গের বই, উপন্যাস",0.853754
9738,বিভূতিভূষণ বন্দ্যোপাধ্যায়,চারটি কিশোর উপন্যাস,শিশু-কিশোর উপন্যাস,0.853608
26118,নবনীতা দেবসেন,দশটি উপন্যাস,"পশ্চিমবঙ্গের বই, উপন্যাস সমগ্র",0.853356
147,সুনীল গঙ্গোপাধ্যায় নীললোহিত,দশটি উপন্যাস,"পশ্চিমবঙ্গের বই, উপন্যাস",0.851618
9778,বিভূতিভূষণ বন্দ্যোপাধ্যায়,গল্প সমগ্র২য় ৫৫টি গল্প,"পশ্চিমবঙ্গের বই, উপন্যাস",0.848524
19515,নারায়ণ গঙ্গোপাধ্যায়,চারমূর্তি,শিশু-কিশোর উপন্যাস,0.846478
29799,তপন বন্দ্যোপাধ্যায়,তিনটি প্রেমের উপন্যাস,"পশ্চিমবঙ্গের বই, উপন্যাস",0.845752


In [ ]:
ft_similar_books('সমাজবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ') #based on weird genres

,author,bangla_title,categories_fixed,similarity_score
34686,প্রফেসর আওরঙ্গজেব,ননমেজর প্রাণিবিদ্যা২ অনার্স ২য় বর্ষ,"প্রাণীবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ",0.991099
90010,রাখী বর্মণ,ধ্রুপদী সমাজ চিন্তা অনার্স ২য় বর্ষ,"সমাজবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ",0.985536
34681,প্রফেসর নওরোজ,জৈব রসায়ন২ অনার্স ২য় বর্ষ,"রসায়ন বিভাগ, অনার্স ২য় বর্ষ",0.982709
76268,সুকেশ চন্দ্র জোয়ারদার,সামষ্টিক অর্থনীতি অনার্স ২য় বর্ষ,"ব্যবস্থাপনা বিভাগ, অনার্স ২য় বর্ষ",0.982431
84088,দিকদর্শন প্রকাশনী সম্পাদক,সমাজকর্ম অনার্স ২য় বর্ষের সাজেশন্স,"সমাজকর্ম বিভাগ, অনার্স ২য় বর্ষ",0.980494
82980,গ্রন্থ কুটির সম্পাদক,গ্রন্থকুটির আবৃতবীজী উদ্ভিদের শ্রেনী বিন্যাস ...,"উদ্ভিদবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ",0.980191
35287,সিরাজুল ইসলাম,প্রাচ্যের রাষ্ট্রচিন্তা অনার্স ২য় বর্ষ বিষয় কো...,"রাষ্ট্রবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ",0.978095
95600,শওকত,সামাজিক পরিসংখ্যান অনার্স ২য় বর্ষ টেক্সট বই সম...,"সমাজবিজ্ঞান বিভাগ, অনার্স ২য় বর্ষ",0.977591
76292,সুশান্ত রায় চৌধুরী,ব্যবসায়ের আইনগত পরিবেশ অনার্স ২য় বর্ষ,"ব্যবস্থাপনা বিভাগ, অনার্স ২য় বর্ষ",0.976131
33013,কামরুল ইসলাম,বাংলাদেশে অর্থনীতি ননমেজর অনার্স ২য় বর্ষ,"অর্থনীতি বিভাগ, অনার্স ২য় বর্ষ",0.975400


In [ ]:
ft_similar_books('হুমায়ুন আহমেদ') #based on writers

,author,bangla_title,categories_fixed,similarity_score
49598,আহমেদ ফারুক,ভালোবাসা আহমেদ,রোমান্টিক উপন্যাস,0.770078
86442,মোশতাক আহমেদ,স্বপ্নস্বর্গ,প্যারাসাইকোলজিকাল উপন্যাস,0.757334
86438,মোশতাক আহমেদ,স্বপ্নখুনি,প্যারাসাইকোলজিকাল উপন্যাস,0.756615
86443,মোশতাক আহমেদ,ছায়াস্বর্গ,প্যারাসাইকোলজিকাল উপন্যাস,0.756201
2042,হুমায়ূন আহমেদ,দেয়াল,ঐতিহাসিক উপন্যাস,0.748924
11074,হুমায়ূন আহমেদ,সম্রাট,থ্রিলার,0.747688
86446,মোশতাক আহমেদ,মায়াস্বর্গ,প্যারাসাইকোলজিকাল উপন্যাস,0.746390
2047,হুমায়ূন আহমেদ,এপিটাফ,সমকালীন উপন্যাস,0.746103
90809,হুমায়ূন আহমেদ,বহুব্রীহি,সমকালীন উপন্যাস,0.745180
87466,হুমায়ূন আহমেদ,ময়ূরাক্ষী,সমকালীন উপন্যাস,0.745018


# 5. Search DB

## 5.1 Difflib

In [ ]:
import difflib
text = 'দয়ল'
difflib.get_close_matches(text, df['bangla_title'].tolist(), n=5)

['দোয়েল', 'দেয়াল', 'দেয়াল', 'দেয়াল', 'বদল']

In [ ]:
def retrieve_matched_entries(dataframe, input_text, title=True):
    close_matches = difflib.get_close_matches(input_text,
                                              dataframe['bangla_title'].tolist() + dataframe['author'].tolist(),
                                              n=50,
                                              cutoff=0.4)

    condition1 = dataframe['bangla_title'].isin(close_matches)
    condition2 = dataframe['author'].isin(close_matches)
    # matched_titles = dataframe.loc[(condition1) | (condition2), ['author', 'bangla_title', 'categories_fixed']]
    matched_titles = dataframe.loc[condition1, ['author', 'bangla_title', 'categories_fixed', 'author_count']]
    matched_author = dataframe.loc[condition2, ['author', 'bangla_title', 'categories_fixed', 'author_count']]

    if title:
        matched_entries = pd.concat([matched_titles, matched_author])
        return matched_entries.sort_values(by='author_count', ascending=False)

    matched_entries = pd.concat([matched_author, matched_titles])
    return matched_entries.sort_values(by='author_count', ascending=False)


In [ ]:
text = 'হুমায়ুন আহমেদ'
retrieved_elem = retrieve_matched_entries(df, text, title=False).sample(20).reset_index(drop=True)

In [ ]:
retrieved_elem

,author,bangla_title,categories_fixed,author_count
0,হুমায়ূন আহমেদ,মিসির আলি অমনিবাস১,সমকালীন উপন্যাস,276
1,হুমায়ূন আহমেদ,ময়ূরাক্ষী,সমকালীন উপন্যাস,276
2,হুমায়ূন আহমেদ,জোছনা ও জননীর গল্প,"রাজনৈতিক, মুক্তিযুদ্ধভিত্তিক উপন্যাস",276
3,হুমায়ূন আহমেদ,পোকা,সমকালীন উপন্যাস,276
4,হুমায়ূন আহমেদ,মিসির আলির চশমা,সমকালীন উপন্যাস,276
5,হুমায়ূন আহমেদ,অয়োময়,সমকালীন গল্প,276
6,হুমায়ূন আহমেদ,হিমু মামা,শিশু-কিশোর উপন্যাস,276
7,হুমায়ূন আহমেদ,হুমায়ূন আহমেদ রচনাবলী ৪,"রচনা সংকলন, সমগ্র",276
8,হুমায়ূন আহমেদ,হিমু রিমান্ডে,সমকালীন উপন্যাস,276
9,হুমায়ূন আহমেদ,সে ও নর্তকী,সমকালীন উপন্যাস,276


## 5.2 Rapidfuzz

In [ ]:
def search_books_fast(search_string, df, limit=5):
    # Use the pandas Series directly for faster matching
    matches = process.extract(search_string, df['bangla_title'], scorer=fuzz.ratio, limit=limit, score_cutoff=0)[0]
    return matches

In [ ]:
class SearchItems:
    def __init__(self, dataframe):
        self.df = dataframe

    def search(self, search_string, title=True, limit=20):
        title_matches = [match[0] for match in
                         process.extract(search_string,
                                         self.df['bangla_title'],
                                         scorer=fuzz.ratio,
                                         limit=limit)]

        author_matches = [match[0] for match in
                         process.extract(search_string,
                                         self.df['author'],
                                         scorer=fuzz.ratio,
                                         limit=limit)]

        condition1 = self.df['bangla_title'].isin(title_matches)
        condition2 = self.df['author'].isin(author_matches)

        matched_titles = self.df.loc[condition1, ['author', 'bangla_title', 'categories_fixed', 'author_count']]
        matched_author = self.df.loc[condition2, ['author', 'bangla_title', 'categories_fixed', 'author_count']]

        if title:
            matched_entries = pd.concat([matched_titles, matched_author])
            return matched_entries.sort_values(by='author_count', ascending=False)

        matched_entries = pd.concat([matched_author, matched_titles])
        return matched_entries.sort_values(by='author_count', ascending=False)


def retrieve_matched_entries(dataframe, input_text, title=True):
    close_matches = difflib.get_close_matches(input_text,
                                              dataframe['bangla_title'].tolist() + dataframe['author'].tolist(),
                                              n=50,
                                              cutoff=0.4)

    condition1 = dataframe['bangla_title'].isin(close_matches)
    condition2 = dataframe['author'].isin(close_matches)
    # matched_titles = dataframe.loc[(condition1) | (condition2), ['author', 'bangla_title', 'categories_fixed']]
    matched_titles = dataframe.loc[condition1, ['author', 'bangla_title', 'categories_fixed', 'author_count']]
    matched_author = dataframe.loc[condition2, ['author', 'bangla_title', 'categories_fixed', 'author_count']]

    if title:
        matched_entries = pd.concat([matched_titles, matched_author])
        return matched_entries.sort_values(by='author_count', ascending=False)

    matched_entries = pd.concat([matched_author, matched_titles])
    return matched_entries.sort_values(by='author_count', ascending=False)

In [ ]:
# Example usage
search_string = "দয়ল"  # A typo in 'Harry Potter'
matches = search_books_fast(search_string, df)

In [ ]:
matches

('দেয়াল', 75.0, 2042)

In [ ]:
search_string = "হুমায়ন"
search = SearchItems(df)
search.search(search_string)

,author,bangla_title,categories_fixed,author_count
87099,হুমায়ূন আহমেদ,সেরা হুমায়ূন,"রচনা সংকলন, সমগ্র",276
90594,হুমায়ুন আজাদ,যাদুকরের মৃত্যু,সমকালীন গল্প,23
90740,হুমায়ুন আজাদ,বুকপকেটে জোনাকী পোকা,শিশুতোষ গ্রন্থ,23
72623,হুমায়ুন আজাদ,বাক্যতত্ত্ব,"ভাষা বিষয়ক গবেষণা, প্রবন্ধ, সমালোচনা",23
90568,হুমায়ুন আজাদ,তুলনামূলক ও ঐতিহাসিক ভাষাবিজ্ঞান,ভাষা বিষয়ক বিবিধ বই,23
...,...,...,...,...
55726,মো সালমানুর রহমান দুর্জয়,মহামায়া,বাংলা কবিতা,1
75544,তৌফিক তুহিন সম্পাদক,হিমিয়ান,বাংলা কবিতা,1
79059,পার্থ চক্রবর্তী,মহামায়া,"পশ্চিমবঙ্গের বই, গল্প",1
85710,হুমায়ুন শেখ,আধুনিক পদ্ধতিতে মসলা ও সুগন্ধি গাছের চাষ,"ফসল, শাক-সবজি চাষ",1


In [ ]:
get_cat = retrieved_elem.loc[0]

In [ ]:
get_cat

,0
author,হুমায়ূন আহমেদ
bangla_title,মিসির আলি অমনিবাস১
categories_fixed,সমকালীন উপন্যাস
author_count,276


In [ ]:

text = get_cat['author'] + " " + get_cat['bangla_title'] + " " + get_cat['categories_fixed']
dv_similar_books(text)

,author,bangla_title,categories_fixed
45525,সুদর্শন দাস,নাট্যগুচ্ছ,"পশ্চিমবঙ্গের বই, নাটক"
2049,হালিমা খাতুন,কিশোর ভুবন ছোট গল্প,শিশু-কিশোর গল্প
85976,শামীম ইসলাম,জীবনের জয় ও অমর বাণী,"উক্তি, বাণী, শ্লোক, প্রবাদ-প্রবচন"
19158,অলকা সরকার কেয়া,বঙ্গবন্ধু তোমার নামে,"মুক্তিযুদ্ধ, ভাষা আন্দোলন, রাজনৈতিক কবিতা"
2185,ধ্রুব এষ,মনে পড়ে এই হেমন্তের রাতে,সমকালীন উপন্যাস
81365,মুহম্মদ মোকাররম হোসায়েন,বিবিক্ত,বাংলা কবিতা
42883,অমিভাত হালদার সম্পাদক,আমি কিংবদন্তি হব,বাংলা কবিতা
35456,ভীষ্মদেব চৌধুরী,জনান্তিকের মুক্তিযুদ্ধ,"সমাজ, সভ্যতা, সংস্কৃতি বিষয়ক প্রবন্ধ"
7392,মঈনুল আহসান সাবের,মানুষ যেখানে যায় না,সমকালীন উপন্যাস
86335,গাজী শরিফুল হাসান,বার্মিংহাম ডায়েরি,ভ্রমণ বিষয়ক স্মৃতি


In [ ]:
text = get_cat['author'] + " " + get_cat['bangla_title'] + " " + get_cat['categories_fixed']
# text = get_cat['categories_fixed']
wv_similar_books(text)[1:]

,author,bangla_title,categories_fixed,similarity_score
11517,হুমায়ূন আহমেদ,মিসির আলি অমনিবাস২,সমকালীন উপন্যাস,0.999995
11574,হুমায়ূন আহমেদ,মিসির আলি আনসলভ্ড,সমকালীন উপন্যাস,0.999992
11627,হুমায়ূন আহমেদ,বাঘবন্দি মিসির আলি,সমকালীন উপন্যাস,0.999992
11649,হুমায়ূন আহমেদ,হিমু মিসির আলি যুগলবন্দি,সমকালীন উপন্যাস,0.999613
11403,হুমায়ূন আহমেদ,মিসির আলির চশমা,সমকালীন উপন্যাস,0.998339
11412,হুমায়ূন আহমেদ,একজন মায়াবতী,সমকালীন উপন্যাস,0.997362
11424,হুমায়ূন আহমেদ,নন্দিত নরকে,সমকালীন উপন্যাস,0.997304
11419,হুমায়ূন আহমেদ,মাতাল হাওয়া,সমকালীন উপন্যাস,0.997107
11629,হুমায়ূন আহমেদ,এইসব দিনরাত্রি,সমকালীন উপন্যাস,0.997029
11566,হুমায়ূন আহমেদ,মীরার গ্রামের বাড়ী,সমকালীন উপন্যাস,0.996825


In [ ]:
df.columns

Index(['author', 'bangla_title', 'categories_fixed', 'combined_feature',
       'tokenized_feature', 'book_vector', 'similarity_score', 'count'],
      dtype='object')